# Notebook 06 — Final Dataset Assembly

Reads per-event strandings data with weather, moon, and plankton features, then aggregates to weekly counts per region for modeling.

**Reads:** `strandings.parquet`, `strandings_with_weather.parquet`, `strandings_with_moon.parquet`, `plankton_imputed_lookup.parquet`  
**Writes:** `data/processed/final_dataset.parquet`

**Prerequisites:** Fixes F1–F3 must be complete before running this notebook.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from se_coast_strandings import make_degrees
from se_coast_strandings.contextual_data.plankton_abundance import assign_region
from se_coast_strandings.transformations import (
    make_dt_col,
    make_cyclic,
    make_cyclic_season,
)
from se_coast_strandings.contextual_data.lunar_phases import add_moon_features

PROCESSED_DIR = Path("../data/processed")

In [2]:
# ── Region tuning ────────────────────────────────────────────────────────────
# Width of each latitude band in decimal degrees. Changing this value and
# re-running notebooks 05 → 06 → 09 → 10 re-partitions the SE coast (32°–38° N)
# into equal-width bands and propagates through the full pipeline.
DEGREES_PER_BAND = 0.5
REGIONS = make_degrees(DEGREES_PER_BAND)

## Load and verify base strandings


In [3]:
base = pd.read_parquet(PROCESSED_DIR / "strandings_with_moon.parquet")
base = base.dropna(subset=["mms_observation_dt", "Latitude", "Longitude"])
base = base.reset_index(drop=True)
print(
    f"Base strandings (with weather + moon): {len(base)} rows, {len(base.columns)} columns"
)

Base strandings (with weather + moon): 2222 rows, 89 columns


## Join weather and moon features


In [4]:
# Weather and moon features already present in strandings_with_moon.parquet
weather_cols = [c for c in base.columns if c.startswith("temperature_2m")]
print(f"Weather columns: {len(weather_cols)}")
print(f"Moon columns: {[c for c in base.columns if 'moon' in c]}")
print(f"After joins: {len(base)} rows, {len(base.columns)} columns")

Weather columns: 26
Moon columns: ['moon_age', 'moon_phase']
After joins: 2222 rows, 89 columns


base["region"] = assign_region(base["Latitude"], regions=REGIONS)
base = base.dropna(subset=["region"])
print(f"After region assignment: {len(base)} rows")
print(base["region"].value_counts())


In [5]:
base["region"] = assign_region(base["Latitude"], regions=REGIONS)
base = base.dropna(subset=["region"])
print(f"After region assignment: {len(base)} rows")
print(base["region"].value_counts())

After region assignment: 2216 rows
region
R9     375
R10    301
R1     269
R6     249
R11    200
R5     179
R7     178
R8     169
R0     131
R3      81
R4      48
R2      36
Name: count, dtype: int64


## Build cyclic features and snap to week


In [6]:
base["week_start"] = base["mms_observation_dt"].dt.to_period("W").dt.start_time

base["month_sin"], base["month_cos"] = make_cyclic(
    base["mms_observation_dt"].dt.month, 12, name="month"
)
base["dayofyear_sin"], base["dayofyear_cos"] = make_cyclic(
    base["mms_observation_dt"].dt.dayofyear, 365, name="dayofyear"
)
base["season_sin"], base["season_cos"] = make_cyclic_season(
    base["mms_observation_dt"], name="season"
)
# Day-of-week of the stranding event (0=Mon, 6=Sun); period=7
base["dayofweek_sin"], base["dayofweek_cos"] = make_cyclic(
    base["mms_observation_dt"].dt.dayofweek, 7, name="dayofweek"
)

## Aggregate to weekly counts per region


In [7]:
day0_max = "temperature_2m_max_0_days_prior"
day0_min = "temperature_2m_min_0_days_prior"
day1_max = "temperature_2m_max_1_days_prior"

if day1_max in base.columns:
    base["temp_delta_day0"] = base[day0_max] - base[day1_max]

# 7-day rolling temperature metrics (days 0–6 prior to stranding)
max_cols_7d = [
    f"temperature_2m_max_{d}_days_prior"
    for d in range(7)
    if f"temperature_2m_max_{d}_days_prior" in base.columns
]
min_cols_7d = [
    f"temperature_2m_min_{d}_days_prior"
    for d in range(7)
    if f"temperature_2m_min_{d}_days_prior" in base.columns
]

base["temp_7day_mean_max"] = base[max_cols_7d].mean(axis=1)
base["temp_7day_max_max"] = base[max_cols_7d].max(axis=1)
base["temp_7day_min_min"] = base[min_cols_7d].min(axis=1)
base["temp_7day_range"] = base["temp_7day_max_max"] - base["temp_7day_min_min"]

print(
    f"7-day temp columns computed from {len(max_cols_7d)} max cols and {len(min_cols_7d)} min cols"
)

# All per-day weather columns for days 1–6 (day 0 handled separately below)
per_day_weather_cols = [
    col
    for d in range(1, 7)
    for col in [
        f"temperature_2m_max_{d}_days_prior",
        f"temperature_2m_min_{d}_days_prior",
        f"temperature_2m_max_{d}_days_prior_delta",
        f"temperature_2m_min_{d}_days_prior_delta",
    ]
    if col in base.columns
]
# Also include day-0 delta and day-0 min delta if present
for col in ["temperature_2m_min_0_days_prior_delta", "temperature_2m_max_0_days_prior_delta"]:
    if col in base.columns and col not in per_day_weather_cols:
        per_day_weather_cols.append(col)

agg_dict = {
    "National Database Number": "count",
    day0_max: ["mean", "max"],  # list keeps MultiIndex for all columns
    day0_min: "mean",
    "moon_age": "mean",
    "temp_delta_day0": "mean",
    "temp_7day_mean_max": "mean",
    "temp_7day_max_max": "mean",
    "temp_7day_min_min": "mean",
    "temp_7day_range": "mean",
    "month_sin": "first",
    "month_cos": "first",
    "dayofyear_sin": "first",
    "dayofyear_cos": "first",
    "season_sin": "first",
    "season_cos": "first",
    "dayofweek_sin": "mean",
    "dayofweek_cos": "mean",
}
for col in per_day_weather_cols:
    agg_dict[col] = "mean"

# Only include columns that exist
agg_dict = {k: v for k, v in agg_dict.items() if k in base.columns}

weekly = base.groupby(["week_start", "region"]).agg(agg_dict).reset_index()

# Flatten MultiIndex columns
weekly.columns = [
    "_".join(filter(None, c)).strip("_") if isinstance(c, tuple) else c
    for c in weekly.columns
]
weekly = weekly.rename(columns={"National Database Number_count": "stranding_count"})
print(f"Weekly aggregated: {len(weekly)} rows")
print(f"Columns: {list(weekly.columns)}")

7-day temp columns computed from 7 max cols and 7 min cols
Weekly aggregated: 1603 rows
Columns: ['week_start', 'region', 'stranding_count', 'temperature_2m_max_0_days_prior_mean', 'temperature_2m_max_0_days_prior_max', 'temperature_2m_min_0_days_prior_mean', 'moon_age_mean', 'temp_delta_day0_mean', 'temp_7day_mean_max_mean', 'temp_7day_max_max_mean', 'temp_7day_min_min_mean', 'temp_7day_range_mean', 'month_sin_first', 'month_cos_first', 'dayofyear_sin_first', 'dayofyear_cos_first', 'season_sin_first', 'season_cos_first', 'dayofweek_sin_mean', 'dayofweek_cos_mean', 'temperature_2m_max_1_days_prior_mean', 'temperature_2m_min_1_days_prior_mean', 'temperature_2m_max_1_days_prior_delta_mean', 'temperature_2m_min_1_days_prior_delta_mean', 'temperature_2m_max_2_days_prior_mean', 'temperature_2m_min_2_days_prior_mean', 'temperature_2m_max_2_days_prior_delta_mean', 'temperature_2m_min_2_days_prior_delta_mean', 'temperature_2m_max_3_days_prior_mean', 'temperature_2m_min_3_days_prior_mean', 'tem

all*weeks = pd.date_range(
start=base["week_start"].min(),
end=base["week_start"].max(),
freq="W-MON",
)
all_regions = [label for label, *, \_ in REGIONS]

full_index = pd.MultiIndex.from_product(
[all_weeks, all_regions], names=["week_start", "region"]
)
full_grid = pd.DataFrame(index=full_index).reset_index()

weekly = full_grid.merge(weekly, on=["week_start", "region"], how="left")
weekly["stranding_count"] = weekly["stranding_count"].fillna(0).astype(int)
print(
f"After zero-fill: {len(weekly)} rows ({len(all_weeks)} weeks \u00d7 {len(all_regions)} regions)"
)


In [8]:
all_weeks = pd.date_range(
    start=base["week_start"].min(),
    end=base["week_start"].max(),
    freq="W-MON",
)
all_regions = [label for label, _, _ in REGIONS]

full_index = pd.MultiIndex.from_product(
    [all_weeks, all_regions], names=["week_start", "region"]
)
full_grid = pd.DataFrame(index=full_index).reset_index()

weekly = full_grid.merge(weekly, on=["week_start", "region"], how="left")
weekly["stranding_count"] = weekly["stranding_count"].fillna(0).astype(int)
print(
    f"After zero-fill: {len(weekly)} rows ({len(all_weeks)} weeks \u00d7 {len(all_regions)} regions)"
)

After zero-fill: 5640 rows (470 weeks × 12 regions)


## Join plankton density lookup


In [9]:
plankton = pd.read_parquet(PROCESSED_DIR / "plankton_imputed_lookup.parquet")
plankton = plankton.rename(columns={"ds": "week_start", "yhat": "plankton_density"})
plankton["week_start"] = pd.to_datetime(plankton["week_start"])
# Prophet uses Sunday-anchored weeks; snap to Monday to match the strandings weekly grid
plankton["week_start"] = plankton["week_start"] - pd.to_timedelta(
    plankton["week_start"].dt.dayofweek, unit="D"
)

weekly = weekly.merge(
    plankton[["week_start", "region", "plankton_density"]],
    on=["week_start", "region"],
    how="left",
)
print(f"Plankton density NaN count: {weekly['plankton_density'].isna().sum()}")

Plankton density NaN count: 1410


## Recompute features for all weeks + forward-fill weather


In [10]:
weekly = weekly.sort_values(["region", "week_start"]).reset_index(drop=True)

# Recompute cyclic and moon features for ALL weeks (including zero-stranding)
weekly["month_sin"], weekly["month_cos"] = make_cyclic(
    weekly["week_start"].dt.month, 12, name="month"
)
weekly["dayofyear_sin"], weekly["dayofyear_cos"] = make_cyclic(
    weekly["week_start"].dt.dayofyear, 365, name="dayofyear"
)
weekly["season_sin"], weekly["season_cos"] = make_cyclic_season(
    weekly["week_start"], name="season"
)
weekly = add_moon_features(weekly, date_col="week_start")

# Forward-fill weather within each region for zero-stranding weeks
# Includes day-0, 7-day rolling metrics, and dayofweek (event-derived; ffill for zero weeks)
weather_feat_cols = [
    c
    for c in weekly.columns
    if c.startswith("temperature_2m")
    or c
    in (
        "temp_delta_day0_mean",
        "temp_7day_mean_max_mean",
        "temp_7day_max_max_mean",
        "temp_7day_min_min_mean",
        "temp_7day_range_mean",
        "dayofweek_sin_mean",
        "dayofweek_cos_mean",
    )
]
weekly[weather_feat_cols] = weekly.groupby("region")[weather_feat_cols].transform(
    lambda g: g.ffill().bfill()
)
print(f"Weather/derived NaN after fill: {weekly[weather_feat_cols].isna().sum().sum()}")
print(f"Forward-filled columns: {weather_feat_cols}")

Weather/derived NaN after fill: 11280
Forward-filled columns: ['temperature_2m_max_0_days_prior_mean', 'temperature_2m_max_0_days_prior_max', 'temperature_2m_min_0_days_prior_mean', 'temp_delta_day0_mean', 'temp_7day_mean_max_mean', 'temp_7day_max_max_mean', 'temp_7day_min_min_mean', 'temp_7day_range_mean', 'dayofweek_sin_mean', 'dayofweek_cos_mean', 'temperature_2m_max_1_days_prior_mean', 'temperature_2m_min_1_days_prior_mean', 'temperature_2m_max_1_days_prior_delta_mean', 'temperature_2m_min_1_days_prior_delta_mean', 'temperature_2m_max_2_days_prior_mean', 'temperature_2m_min_2_days_prior_mean', 'temperature_2m_max_2_days_prior_delta_mean', 'temperature_2m_min_2_days_prior_delta_mean', 'temperature_2m_max_3_days_prior_mean', 'temperature_2m_min_3_days_prior_mean', 'temperature_2m_max_3_days_prior_delta_mean', 'temperature_2m_min_3_days_prior_delta_mean', 'temperature_2m_max_4_days_prior_mean', 'temperature_2m_min_4_days_prior_mean', 'temperature_2m_max_4_days_prior_delta_mean', 'temp

## Add lag features


In [11]:
weekly["stranding_count_lag_1"] = weekly.groupby("region")["stranding_count"].shift(1)
weekly["stranding_count_lag_52"] = weekly.groupby("region")["stranding_count"].shift(52)
# Drop first 52 weeks per region (insufficient lag history)
weekly = weekly.dropna(subset=["stranding_count_lag_52"]).reset_index(drop=True)
print(f"After lag drop: {len(weekly)} rows")

After lag drop: 5016 rows


## Encode region and save


In [12]:
region_dummies = pd.get_dummies(weekly["region"], prefix="region", drop_first=False)
weekly = pd.concat([weekly, region_dummies], axis=1)

# Schema validation
REQUIRED_COLS = [
    "week_start",
    "region",
    "stranding_count",
    "month_sin",
    "month_cos",
    "dayofyear_sin",
    "dayofyear_cos",
    "season_sin",
    "season_cos",
    "dayofweek_sin_mean",
    "dayofweek_cos_mean",
    "moon_age",
    "plankton_density",
    "temp_7day_mean_max_mean",
    "temp_7day_max_max_mean",
    "temp_7day_min_min_mean",
    "temp_7day_range_mean",
    "stranding_count_lag_1",
    "stranding_count_lag_52",
]
for col in REQUIRED_COLS:
    assert col in weekly.columns, f"Missing required column: {col}"

weekly.to_parquet(PROCESSED_DIR / "final_dataset.parquet", index=False)
print(
    f"Saved final_dataset.parquet: {len(weekly):,} rows × {len(weekly.columns)} columns"
)
print(f"\nColumns: {list(weekly.columns)}")
weekly.head()

Saved final_dataset.parquet: 5,016 rows × 67 columns

Columns: ['week_start', 'region', 'stranding_count', 'temperature_2m_max_0_days_prior_mean', 'temperature_2m_max_0_days_prior_max', 'temperature_2m_min_0_days_prior_mean', 'moon_age_mean', 'temp_delta_day0_mean', 'temp_7day_mean_max_mean', 'temp_7day_max_max_mean', 'temp_7day_min_min_mean', 'temp_7day_range_mean', 'month_sin_first', 'month_cos_first', 'dayofyear_sin_first', 'dayofyear_cos_first', 'season_sin_first', 'season_cos_first', 'dayofweek_sin_mean', 'dayofweek_cos_mean', 'temperature_2m_max_1_days_prior_mean', 'temperature_2m_min_1_days_prior_mean', 'temperature_2m_max_1_days_prior_delta_mean', 'temperature_2m_min_1_days_prior_delta_mean', 'temperature_2m_max_2_days_prior_mean', 'temperature_2m_min_2_days_prior_mean', 'temperature_2m_max_2_days_prior_delta_mean', 'temperature_2m_min_2_days_prior_delta_mean', 'temperature_2m_max_3_days_prior_mean', 'temperature_2m_min_3_days_prior_mean', 'temperature_2m_max_3_days_prior_delta

,week_start,region,stranding_count,temperature_2m_max_0_days_prior_mean,temperature_2m_max_0_days_prior_max,temperature_2m_min_0_days_prior_mean,moon_age_mean,temp_delta_day0_mean,temp_7day_mean_max_mean,temp_7day_max_max_mean,...,region_R10,region_R11,region_R2,region_R3,region_R4,region_R5,region_R6,region_R7,region_R8,region_R9
0,2017-01-02,R0,0,20.8,20.8,14.4,NaN,4.4,16.228571,21.50,...,False,False,False,False,False,False,False,False,False,False
1,2017-01-09,R0,0,20.8,20.8,14.4,NaN,4.4,16.228571,21.50,...,False,False,False,False,False,False,False,False,False,False
2,2017-01-16,R0,2,19.8,20.5,15.1,22.132944,-1.4,20.835714,22.95,...,False,False,False,False,False,False,False,False,False,False
3,2017-01-23,R0,0,19.8,20.5,15.1,NaN,-1.4,20.835714,22.95,...,False,False,False,False,False,False,False,False,False,False
4,2017-01-30,R0,1,20.1,20.1,11.1,5.102356,0.0,15.085714,20.10,...,False,False,False,False,False,False,False,False,False,False
